# STEP 14 - the Bengali.ai normalizer as a defense baseline

`bnunicodenormalizer` (Ansary et al., LREC-COLING 2024) is the published Indic Unicode
normalizer. It is not a security tool and its paper never mentions zero-width characters,
but the library removes them, so it defends against our attack by accident. That makes it
the strongest available baseline and it has to be in the comparison.

Measured locally already: it stops all three attacks completely, and it is smarter than
strip-all because it keeps the joiner in RAB. But on 57 real in-the-wild words it still
destroys 10, where render-equivalence destroys none.

This notebook puts it on the same footing as every other defense in the paper:

1. **Security** - does it restore the exact clean text for each attack?
2. **Collateral** - on the same ~340 real comments that already contain zero-width
   characters, how many does each defense visibly change?
3. **Cost on ordinary text** - a full normalizer also rewrites spelling. We measure how
   many untouched comments it alters, which strip-all and render-equivalence never do.

**No GPU needed.** Set the accelerator to **None** and save your quota.
Internet **ON**, data **`naurosromim/bdshs`**. About five minutes.


In [ ]:
!pip install -q uharfbuzz bnunicodenormalizer "emoji==1.7.0" regex ftfy
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
import sys, os, glob, random, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd
from bnunicodenormalizer import Normalizer

OUT = "/kaggle/working"
R = ra.Renderer(ra.fetch_fonts(f"{OUT}/fonts"))
_bn = Normalizer()

def bnorm(text):
    """bnunicodenormalizer works one word at a time and can return None."""
    out = []
    for w in text.split():
        try:
            r = _bn(w)
            out.append(r["normalized"] if r and r.get("normalized") else w)
        except Exception:
            out.append(w)
    return " ".join(out)

paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
rtr, rte = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in rtr.columns if rtr[c].dtype == object]
TEXT = max(obj, key=lambda c: rtr[c].astype(str).str.len().mean())
LABEL = [c for c in rtr.columns if c != TEXT and rtr[c].dropna().nunique() == 2][0]
def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)
train, test = prep(rtr), prep(rte)
HATE = 1
print("train", len(train), "| test", len(test))

# identical attack sets and seeds as every earlier step
rng = random.Random(0)
hate_txt = [test.text[i] for i in sorted(rng.sample(test.index[test.label == HATE].tolist(), 1500))]
ATK = {
    "gated_inert_ALL":           [R.attack(t, "ALL", mode="inert", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_adaptive_hasant_ALL": [R.attack(t, "ALL", mode="hasant", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_both_ALL":            [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(hate_txt)],
}
DEFENSES = {
    "nfc": ra.defend_nfc,
    "strip_all": ra.defend_strip_all,
    "position_aware": ra.defend_position_aware,
    "bnunicodenormalizer": bnorm,
    "render_equivalence": R.defend_render_equivalence,
}
print("defenses:", list(DEFENSES))

In [ ]:
# 1. SECURITY - does the defense hand back the exact clean text?
# If it does, the model sees precisely what it saw before the attack, so evasion is 0
# for the same reason it is 0 for render-equivalence. No GPU needed to establish that.
rows = []
for a, texts in ATK.items():
    for d, fn in DEFENSES.items():
        t0 = time.time()
        exact = float(np.mean([fn(x) == c for x, c in zip(texts, hate_txt)]))
        left = float(np.mean([any(z in fn(x) for z in ra.ZW) for x in texts]))
        rows.append({"attack": a, "defense": d,
                     "exact_clean_recovery": round(exact, 4),
                     "zero_width_remaining": round(left, 4),
                     "ms_per_comment": round((time.time() - t0) / len(texts) * 1000, 3)})
        print(rows[-1])
sec = pd.DataFrame(rows); sec.to_csv(f"{OUT}/step14_security.csv", index=False)

In [ ]:
# 2. COLLATERAL - the same basis as the paper: real comments that already contain
#    zero-width characters, and whether the defense changes how they are drawn.
alltxt = train.text.tolist() + test.text.tolist()
zw_docs = [t for t in alltxt if any(z in t for z in ra.ZW)]
print(f"real comments containing zero-width characters: {len(zw_docs)} of {len(alltxt)}")
rows = []
for d, fn in DEFENSES.items():
    changed = sum(fn(t) != t for t in zw_docs)
    visible = sum(not R.same(t, fn(t)) for t in zw_docs)
    rows.append({"defense": d, "docs_with_zw": len(zw_docs),
                 "codepoints_changed": changed, "rendering_changed_any_font": visible})
    print(rows[-1])
coll = pd.DataFrame(rows); coll.to_csv(f"{OUT}/step14_collateral.csv", index=False)

In [ ]:
# 3. COST ON ORDINARY TEXT - a full normalizer also rewrites spelling. strip-all and
#    render-equivalence touch nothing that has no zero-width character; this shows what
#    adopting a general normalizer as a security control actually costs.
sample = test.text.tolist()
rows = []
for d, fn in DEFENSES.items():
    changed = sum(fn(t) != t for t in sample)
    visible = sum(not R.same(t, fn(t)) for t in sample)
    rows.append({"defense": d, "clean_comments": len(sample),
                 "codepoints_changed": changed,
                 "pct_codepoints_changed": round(changed / len(sample), 4),
                 "rendering_changed": visible,
                 "pct_rendering_changed": round(visible / len(sample), 4)})
    print(rows[-1])
cost = pd.DataFrame(rows); cost.to_csv(f"{OUT}/step14_clean_text_cost.csv", index=False)

In [ ]:
pd.set_option("display.width", 250)
print("="*94); print("STEP 14 SUMMARY - bnunicodenormalizer as a defense baseline"); print("="*94)
print("-- security: exact clean-text recovery --")
print(sec.pivot(index="defense", columns="attack", values="exact_clean_recovery").to_string())
print("\n-- zero-width characters left behind --")
print(sec.pivot(index="defense", columns="attack", values="zero_width_remaining").to_string())
print("\n-- collateral on real comments that already contain zero-width --")
print(coll.to_string(index=False))
print("\n-- cost on ordinary clean text --")
print(cost.to_string(index=False))
print("="*94)